# 21 · GSE135779 · scRNA_seq · metadata

Reads the series matrix. Writes `data/run_artifacts/GSE135779/metadata.rds`.

GEO records, per sample: age, age group (Children or Adults) and group (SLE or HD, healthy donor).
Only children are kept. The clinical data come from the paper's Supplementary Table 1b (sheet
`ST1b-Clinical information`, file `41590_2020_743_MOESM3_ESM.xlsx`), one row per individual, joined to
GEO by donor name (`cSLE1`, `cHD1`, …).

In [1]:
source("../src/paths.R")
suppressMessages({library(GEOquery); library(Biobase)})
pd <- pData(suppressMessages(getGEO(filename = raw("GSE135779", "GSE135779_series_matrix.txt.gz"), getGPL = FALSE)))
grep(":ch1$", names(pd), value = TRUE)

[1] "age group:ch1" "age:ch1"       "groups:ch1"

In [2]:
meta <- data.frame(sample = rownames(pd), donor = sub(" \\[.*", "", pd$title),
                   library = sub(".*\\[(.*)\\]", "\\1", pd$title),
                   age = as.numeric(pd[["age:ch1"]]), age_group = pd[["age group:ch1"]],
                   sle = as.numeric(pd[["groups:ch1"]] == "SLE"), row.names = rownames(pd))
table(age_group = meta$age_group, sle = meta$sle)
meta <- meta[meta$age_group == "Children", ]
c(children = nrow(meta), SLE = sum(meta$sle), healthy = sum(meta$sle == 0))
tapply(meta$age, meta$sle, summary)

          sle
age_group   0  1
  Adult     5  7
  Children 11 33

children      SLE  healthy 
      44       33       11

$`0`
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
   7.00    8.00   13.00   12.27   15.00   18.00 

$`1`
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
  10.00   15.00   16.00   15.85   17.00   19.00 


**Result.** 44 children: 33 with SLE (aged 10–19) and 11 healthy (aged 7–18). The 12 adults are dropped.

## Supplementary Table 1b

The sheet has three title rows; the column names are on row 4. `ND` is a missing value. Every column
is shown before any is used.

In [3]:
st <- as.data.frame(readxl::read_excel(raw("GSE135779", "41590_2020_743_MOESM3_ESM.xlsx"),
                                       sheet = "ST1b-Clinical information ", skip = 3, na = c("ND", "NA", "")))
dim(st)
names(st)

[1] 58 58

[1] "Names"              "Groups"             "Batch"             
 [4] "Age"                "Gender"             "Race"              
 [7] "Ethnicity"          "SLEDAI"             "Physician Notes"   
[10] "Collection_year"    "Symptoms"           "MDG"               
[13] "Arthritis"          "Rash"               "Proteinuria"       
[16] "Hematuria"          "Urinary Casts"      "Pyuria"            
[19] "dsDNA"              "low_com"            "fever"             
[22] "pleurisy"           "vasculitis"         "Alopecia"          
[25] "Leukopenia"         "MMF"                "OS"                
[28] "MTX"                "Plaquenil"          "Neph_all"          
[31] "Proliferative_Neph" "Memb_Neph"          "Nephrology_Class"  
[34] "Neph_class"         "ESR"                "WBC"               
[37] "RBC"                "MONOCYTE_per"       "NEUTROPHIL_per"    
[40] "LYMPHOCYTE_per"     "HGB"                "HCT"               
[43] "PLATELETS"          "NEU_ABS"            "LYM_ABS"           
[46] "CREATININE"         "ALBUMIN"            "DSDNA_ratio"       
[49] "DSDNA"              "C3"                 "C4"                
[52] "ALT"                "AST"                "ALD"               
[55] "LDH"                "MSK"                "KIDNEY"            
[58] "SERUM"

**Check.** Every child in GEO has a row in the table, and no row is matched twice.

In [4]:
c(children_in_GEO = nrow(meta), matched = sum(meta$donor %in% st$Names), duplicated_names = sum(duplicated(st$Names)))
stopifnot(all(meta$donor %in% st$Names), !anyDuplicated(st$Names))

children_in_GEO          matched duplicated_names 
              44               44                0

**Result.** All 44 children match a row of Supplementary Table 1b, and no name appears twice.

**Check.** Age and group agree between GEO and the table.

In [5]:
tb <- st[match(meta$donor, st$Names), ]
c(age_agrees = sum(as.numeric(tb$Age) == meta$age, na.rm = TRUE),
  group_agrees = sum((tb$Groups == "cSLE") == (meta$sle == 1)))

age_agrees group_agrees 
          44           44

**Result.** Age and group agree between GEO and the table for all 44 children.

In [6]:
meta <- cbind(meta, tb[, setdiff(names(tb), c("Names", "Age", "Groups"))])
sle <- meta[meta$sle == 1, ]
str(sle[, c("SLEDAI", "MDG", "Neph_all", "Proliferative_Neph", "Neph_class", "MMF", "OS", "MTX", "Plaquenil",
            "WBC", "NEU_ABS", "LYM_ABS", "PLATELETS", "DSDNA", "C3", "C4")])

'data.frame':	33 obs. of  16 variables:
 $ SLEDAI            : num  0 0 4 0 0 4 0 2 18 8 ...
 $ MDG               : num  0 0 0 0 0 1 0 0 1 2 ...
 $ Neph_all          : num  1 0 1 0 1 1 1 0 1 1 ...
 $ Proliferative_Neph: num  NA NA NA 0 1 1 0 0 NA NA ...
 $ Neph_class        : chr  "2,5" "0" "5" "0" ...
 $ MMF               : num  1 1 1 1 0 1 1 0 0 0 ...
 $ OS                : num  0 0 1 0 0 1 1 1 0 0 ...
 $ MTX               : num  0 0 0 1 0 0 0 0 0 0 ...
 $ Plaquenil         : num  0 1 1 1 0 1 1 1 0 0 ...
 $ WBC               : num  4.8 3.6 4.9 4.9 10 7.4 3.4 NA 9.8 4.5 ...
 $ NEU_ABS           : num  2.03 1.61 2.83 3.38 6.95 5.58 1.61 NA 7.67 3.93 ...
 $ LYM_ABS           : num  2.1 1.54 1.43 0.98 2.11 1.18 1.35 NA 1.35 0.38 ...
 $ PLATELETS         : num  280 242 174 135 309 240 235 NA 322 195 ...
 $ DSDNA             : chr  "none detected" "none detected" "1\\40" "none detected" ...
 $ C3                : num  109 108 58 79 143 89 139 NA 67 35 ...
 $ C4                : num  26.8 2

**Note.** `DSDNA` here is a titer (for example `1\640`, or `none detected`). GSE65391 records anti-dsDNA
in different units. The two cannot be treated as the same variable.

In [7]:
table(DSDNA = sle$DSDNA, useNA = "ifany")

DSDNA
        1\\40        1\\640         1\\80      detected none detected 
            3             2             1            12            11 
         <NA> 
            4 

In [8]:
saveRDS(meta, art("GSE135779", "metadata.rds"))